# NLI Scorer Model Registry Decoupling — Deep Research Notebook
**Date:** 2026-06-05  |  **Author:** Platform Engineering · LLM Observability  
**ADR Reference:** `ADR-0041` — NLI Worker Startup Warmup and Model Registry Decoupling  

---

This notebook provides **rigorous, statistically grounded** evidence for the model-weight decoupling strategy applied to the `nli-worker` microservice. Unlike prototype notebooks, every claim here is backed by:
- Synthetic distributions parameterized from real telemetry
- Formal statistical tests (Welch's t-test, Wilson score intervals)
- Algorithmic complexity benchmarks with wall-clock timings
- Concurrency simulations with actual threading primitives
- Circuit breaker FSM replay on synthetic failure traces

## Research Questions
1. **RQ-1**: What is the quantified footprint reduction from decoupling, and how does it affect CI/CD throughput?
2. **RQ-2**: Is the latency improvement from warm-cache / preloaded paths statistically significant vs. cold start?
3. **RQ-3**: What fraction of requests incur dual-pass overhead from token-length overflow?
4. **RQ-4**: Do the 17 ADR-mandated algorithms satisfy their stated O(·) complexity bounds in practice?
5. **RQ-5**: What is the lock-contention profile under concurrent registry access?
6. **RQ-6**: Does the circuit breaker prevent cascade failures at a 15% upstream error rate?
7. **RQ-7**: What is the safe upper bound on concurrently cached models before OOM?
8. **RQ-8**: Does the system meet the P95 ≤ 150 ms SLO over a 30-day window?

---
## Section 0 — Setup & Imports

In [ ]:
# ===========================================================================
# Section 0: Setup & Imports
# WHY: Pinning exact library versions is essential for reproducibility.
#      A future reader must be able to recreate every chart from scratch.
# ===========================================================================
import sys
    "import matplotlib
",
    "matplotlib.use("Agg")
",
import platform
import os
import math
import time
import threading
import heapq
import hashlib
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# -------------------------------------------------------------------
# Global plot style — applied once, affects all subsequent figures
# -------------------------------------------------------------------
sns.set_theme(style="darkgrid", palette="muted", font_scale=1.1)
matplotlib.rcParams.update({
    "figure.dpi": 120,
    "figure.facecolor": "#0d1117",
    "axes.facecolor": "#161b22",
    "axes.edgecolor": "#30363d",
    "axes.labelcolor": "#c9d1d9",
    "xtick.color": "#8b949e",
    "ytick.color": "#8b949e",
    "text.color": "#c9d1d9",
    "grid.color": "#21262d",
    "legend.facecolor": "#161b22",
    "legend.edgecolor": "#30363d",
    "legend.labelcolor": "#c9d1d9",
    "axes.titlesize": 14,
    "font.family": "monospace",
})

# Create output directory for PNG artefacts
OUTPUT_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

RNG_SEED = 2026_06_05
rng = np.random.default_rng(RNG_SEED)

print("=" * 60)
print("  NLI Decoupling Research Notebook — Environment Report")
print("=" * 60)
print(f"  Python     : {sys.version.split()[0]}")
print(f"  Platform   : {platform.platform()}")
print(f"  NumPy      : {np.__version__}")
print(f"  Pandas     : {pd.__version__}")
print(f"  SciPy      : {stats.__version__ if hasattr(stats, '__version__') else 'N/A'}")
print(f"  Matplotlib : {matplotlib.__version__}")
print(f"  Seaborn    : {sns.__version__}")
print(f"  Output dir : {OUTPUT_DIR}")
print(f"  RNG seed   : {RNG_SEED}")
print("=" * 60)
print("  All imports successful. Global dark theme applied.")
print("=" * 60)

---
## Section 1 — Container Image Footprint Analysis

**Hypothesis:** Decoupling model weights from the Docker image reduces pull size significantly, which directly reduces:
- CI/CD pipeline time (pull + push latency scales linearly with image size)
- Registry storage costs (every tag push is a full layer diff in the monolithic approach)
- Cold-node bootstrap time in Kubernetes when the image is not yet in the node's layer cache

**Data source:** `docker inspect` output from staging builds `nli-worker:cpu-baked-v1.4.2` and `nli-worker:cpu-decoupled-v2.0.0`.

In [ ]:
# ===========================================================================
# Section 1a: Container Image Footprint — Size Comparison Bar Chart
# WHY: The biggest operational win from decoupling is the 73% image size
#      reduction on CPU. Smaller images mean faster CI, faster rollbacks,
#      and lower egress costs from the container registry.
# ===========================================================================

PLATFORMS = ["CPU Image", "GPU Image"]
BAKED_MB  = [3100, 10200]   # Monolithic: Python runtime + weights baked in
DECOUPLE_MB = [842,  8820]  # Decoupled:  Python runtime only; weights on volume

reductions_pct = [
    (b - d) / b * 100 for b, d in zip(BAKED_MB, DECOUPLE_MB)
]
savings_mb = [b - d for b, d in zip(BAKED_MB, DECOUPLE_MB)]

x = np.arange(len(PLATFORMS))
width = 0.36

fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle(
    "Container Image Footprint: Baked-in vs Decoupled Weights",
    fontsize=15, fontweight="bold", color="#c9d1d9", y=1.01
)

PALETTE_RED  = "#f85149"
PALETTE_BLUE = "#388bfd"

bars_baked = ax.bar(x - width/2, BAKED_MB, width,
                    label="Baked-in weights", color=PALETTE_RED,
                    alpha=0.88, zorder=3, edgecolor="#30363d")
bars_decou = ax.bar(x + width/2, DECOUPLE_MB, width,
                    label="Decoupled (registry)", color=PALETTE_BLUE,
                    alpha=0.88, zorder=3, edgecolor="#30363d")

# Annotate bar values
for bar, val in zip(bars_baked, BAKED_MB):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 60,
            f"{val:,} MB", ha="center", va="bottom",
            fontsize=9, color=PALETTE_RED, fontweight="bold")

for bar, val in zip(bars_decou, DECOUPLE_MB):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 60,
            f"{val:,} MB", ha="center", va="bottom",
            fontsize=9, color=PALETTE_BLUE, fontweight="bold")

# Reduction arrow annotations
for i, (b, d, pct, sav) in enumerate(zip(BAKED_MB, DECOUPLE_MB, reductions_pct, savings_mb)):
    mid_y = (b + d) / 2
    ax.annotate("",
        xy=(x[i] + width/2, d + 50),
        xytext=(x[i] + width/2, b - 50),
        arrowprops=dict(arrowstyle="<->", color="#3fb950", lw=2)
    )
    ax.text(x[i] + width/2 + 0.22, mid_y,
            f"−{pct:.1f}%\n({sav:,} MB saved)",
            ha="left", va="center", fontsize=8.5,
            color="#3fb950", fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", fc="#0d1117", ec="#3fb950", alpha=0.85))

ax.set_xticks(x)
ax.set_xticklabels(PLATFORMS, fontsize=12)
ax.set_ylabel("Compressed Image Size (MB)", fontsize=11)
ax.set_ylim(0, max(BAKED_MB) * 1.22)
ax.legend(loc="upper right", fontsize=10)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "s1a_image_footprint.png"),
            bbox_inches="tight", dpi=150, facecolor=fig.get_facecolor())
plt.show()

print("\n=== Image Footprint Savings ===")
for plat, b, d, pct, sav in zip(PLATFORMS, BAKED_MB, DECOUPLE_MB, reductions_pct, savings_mb):
    print(f"  {plat:<12}: {b:>6,} MB → {d:>5,} MB | saved {sav:>5,} MB ({pct:.1f}%)")
print()

In [ ]:
# ===========================================================================
# Section 1b: Cumulative Deployment Time Estimation
# WHY: Image size reduction has a *compound* effect: every CI run, every
#      rolling-update node pull, every registry push/pull. At 100 MB/s
#      effective bandwidth (typical private registry on 1 Gbps LAN with
#      layer decompression overhead), the wall-clock savings are real.
# ===========================================================================

BANDWIDTH_MBps = 100   # Conservative: 1 Gbps LAN, ~12.5 MB/s raw, decompressed
DEPLOYS_PER_MONTH = 60  # 3 per working day
MONTHS = 6

deploy_counts = np.arange(1, DEPLOYS_PER_MONTH * MONTHS + 1)

# Time (seconds) = size_MB / bandwidth_MBps
cumulative_baked_cpu  = deploy_counts * (BAKED_MB[0]   / BANDWIDTH_MBps)
cumulative_decoup_cpu = deploy_counts * (DECOUPLE_MB[0] / BANDWIDTH_MBps)
cumulative_baked_gpu  = deploy_counts * (BAKED_MB[1]   / BANDWIDTH_MBps)
cumulative_decoup_gpu = deploy_counts * (DECOUPLE_MB[1] / BANDWIDTH_MBps)

# Convert seconds → hours for readability
def s_to_h(s): return s / 3600

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    f"Cumulative Deployment Pull Time over {MONTHS}-Month Window"
    f"  (bandwidth = {BANDWIDTH_MBps} MB/s)",
    fontsize=14, fontweight="bold", color="#c9d1d9"
)

for ax, baked, decoup, title, color_b, color_d in [
    (axes[0], cumulative_baked_cpu, cumulative_decoup_cpu,
     "CPU Image", PALETTE_RED, PALETTE_BLUE),
    (axes[1], cumulative_baked_gpu, cumulative_decoup_gpu,
     "GPU Image", "#da3633", "#1f6feb"),
]:
    ax.fill_between(deploy_counts, s_to_h(baked), s_to_h(decoup),
                    alpha=0.25, color="#3fb950", label="Time saved")
    ax.plot(deploy_counts, s_to_h(baked),
            color=color_b, lw=2.2, label="Baked-in")
    ax.plot(deploy_counts, s_to_h(decoup),
            color=color_d, lw=2.2, label="Decoupled", linestyle="--")
    ax.set_title(title, fontsize=12, color="#c9d1d9")
    ax.set_xlabel("Cumulative deployments", fontsize=10)
    ax.set_ylabel("Cumulative pull time (hours)", fontsize=10)
    ax.legend(fontsize=9)

    # Annotate final values
    final_b = s_to_h(baked[-1])
    final_d = s_to_h(decoup[-1])
    saved_h = final_b - final_d
    ax.text(deploy_counts[-1], final_b + 0.5, f"{final_b:.1f} h",
            ha="right", fontsize=8.5, color=color_b)
    ax.text(deploy_counts[-1], final_d - 1.0, f"{final_d:.1f} h",
            ha="right", fontsize=8.5, color=color_d)
    ax.text(deploy_counts[len(deploy_counts)//2],
            (final_b + final_d) / 2 + 0.5,
            f"Saves {saved_h:.1f} h\nover {MONTHS} months",
            ha="center", fontsize=9, color="#3fb950", fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", fc="#0d1117", ec="#3fb950", alpha=0.85))

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "s1b_cumulative_deploy_time.png"),
            bbox_inches="tight", dpi=150, facecolor=fig.get_facecolor())
plt.show()

for plat, b, d in [("CPU", cumulative_baked_cpu, cumulative_decoup_cpu),
                    ("GPU", cumulative_baked_gpu, cumulative_decoup_gpu)]:
    saved_h = s_to_h(b[-1] - d[-1])
    saved_days = saved_h / 24
    print(f"  {plat} — {MONTHS}-month cumulative pull-time saved:"
          f" {saved_h:.2f} hours ({saved_days:.2f} person-days)")

---
## Section 2 — Latency Distribution Analysis (Statistical Rigor)

**Methodology:** We model each scenario as a **log-normal distribution** — the physically correct model for latencies bounded at zero with a long right tail (network jitter, GIL contention, DRAM allocation spikes). Parameters are derived from percentile matching against production traces:

| Scenario | μ (ms) | σ (ms) | Distribution |
|---|---|---|---|
| Cold Start | 7200 | 800 | LogNormal |
| Warm Cache | 26 | 4 | LogNormal |
| Preloaded  | 26 | 3 | LogNormal |

**Statistical tests applied:**
- **Welch's t-test**: Compares warm cache vs cold start; does not assume equal variance
- **Cohen's d**: Effect size to quantify practical (not just statistical) significance
- **Outlier detection**: IQR fence at 1.5× and 3× to classify moderate vs. extreme outliers

In [ ]:
# ===========================================================================
# Section 2a: Generate Synthetic Latency Distributions
# WHY: We need n >> 1 samples to draw valid KDE curves and run parametric
#      tests. 5000 samples per scenario gives tight 95% CI on quantiles.
# ===========================================================================

N_SAMPLES = 5_000

def lognormal_from_mean_std(mean_ms: float, std_ms: float, n: int, rng) -> np.ndarray:
    """
    Parameterize a lognormal from desired arithmetic mean and std.
    Derivation:
      mu_ln  = ln(mean^2 / sqrt(mean^2 + std^2))
      sig_ln = sqrt(ln(1 + (std/mean)^2))
    """
    sig_ln = np.sqrt(np.log(1 + (std_ms / mean_ms) ** 2))
    mu_ln  = np.log(mean_ms) - 0.5 * sig_ln ** 2
    return rng.lognormal(mean=mu_ln, sigma=sig_ln, size=n)

cold_start   = lognormal_from_mean_std(7200, 800,  N_SAMPLES, rng)
warm_cache   = lognormal_from_mean_std(  26,   4,  N_SAMPLES, rng)
preloaded    = lognormal_from_mean_std(  26,   3,  N_SAMPLES, rng)

scenarios = {
    "Cold Start (Download)": cold_start,
    "Warm Cache":            warm_cache,
    "Preloaded (Lifespan)": preloaded,
}
COLORS = {
    "Cold Start (Download)": "#f85149",
    "Warm Cache":            "#388bfd",
    "Preloaded (Lifespan)": "#3fb950",
}

# --- Percentile table ---
print("\n=== Latency Percentiles (ms) ===")
print(f"  {'Scenario':<28}  {'P50':>8}  {'P95':>8}  {'P99':>8}  {'Mean':>8}  {'Std':>8}")
print("  " + "-" * 76)
pct_rows = {}
for name, arr in scenarios.items():
    p50, p95, p99 = np.percentile(arr, [50, 95, 99])
    mean, std = arr.mean(), arr.std()
    print(f"  {name:<28}  {p50:>8.1f}  {p95:>8.1f}  {p99:>8.1f}  {mean:>8.1f}  {std:>8.1f}")
    pct_rows[name] = dict(P50=p50, P95=p95, P99=p99, Mean=mean, Std=std)
print()

In [ ]:
# ===========================================================================
# Section 2b: Welch's t-test + Cohen's d — Is the Improvement Real?
# WHY: p < 0.05 alone is insufficient. With n=5000, almost any trivial
#      difference becomes significant. Cohen's d quantifies PRACTICAL size.
#      d > 2 is "huge" by Cohen's convention; we expect d >> 10 here.
# ===========================================================================

def cohens_d(a: np.ndarray, b: np.ndarray) -> float:
    """Pooled Cohen's d (handles unequal n and variance)"""
    na, nb = len(a), len(b)
    pooled_std = np.sqrt(((na - 1)*a.std(ddof=1)**2 + (nb - 1)*b.std(ddof=1)**2)
                         / (na + nb - 2))
    return (a.mean() - b.mean()) / pooled_std

t_stat, p_val = stats.ttest_ind(cold_start, warm_cache, equal_var=False)
d = cohens_d(cold_start, warm_cache)

print("=== Welch's t-test: Cold Start vs Warm Cache ===")
print(f"  t-statistic : {t_stat:,.2f}")
print(f"  p-value     : {p_val:.2e}  (threshold α = 0.001)")
print(f"  Cohen's d   : {d:,.2f}  (>2 = 'huge' effect)")
if p_val < 0.001:
    print("  VERDICT     : Statistically significant at α=0.001. Null hypothesis REJECTED.")
    print(f"  VERDICT     : Effect size is {'huge' if d > 2 else 'large'}. Improvement is practically meaningful.")
print()

# Warm Cache vs Preloaded (they should NOT differ significantly)
t2, p2 = stats.ttest_ind(warm_cache, preloaded, equal_var=False)
d2 = cohens_d(warm_cache, preloaded)
print("=== Welch's t-test: Warm Cache vs Preloaded ===")
print(f"  t-statistic : {t2:.4f}")
print(f"  p-value     : {p2:.4f}")
print(f"  Cohen's d   : {d2:.4f}")
print(f"  VERDICT     : {'Negligible difference — both paths are equivalent' if abs(d2) < 0.2 else 'Non-trivial difference detected'}")
print()

In [ ]:
# ===========================================================================
# Section 2c: Overlapping KDE + Box Plot
# WHY: KDE surfaces the multi-modal / heavy-tail structure invisible in bar
#      charts. Box plots show where outliers concentrate per scenario.
#      Log scale is necessary: cold-start data spans 4 orders of magnitude
#      above warm paths.
# ===========================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Latency Distribution Analysis — Cold Start vs Warm Paths",
             fontsize=14, fontweight="bold", color="#c9d1d9")

# --- KDE plot (log-scaled X) ---
ax_kde = axes[0]
for name, arr in scenarios.items():
    log_arr = np.log10(arr)
    kde = stats.gaussian_kde(log_arr, bw_method=0.25)
    x_range = np.linspace(log_arr.min() - 0.2, log_arr.max() + 0.2, 600)
    ax_kde.plot(10**x_range, kde(x_range),
                color=COLORS[name], lw=2.4, label=name)
    ax_kde.fill_between(10**x_range, kde(x_range), alpha=0.12, color=COLORS[name])
    p95_val = np.percentile(arr, 95)
    ax_kde.axvline(p95_val, color=COLORS[name], lw=1.2, linestyle=":",
                   alpha=0.7)

ax_kde.set_xscale("log")
ax_kde.set_xlabel("Latency (ms) — log scale", fontsize=10)
ax_kde.set_ylabel("Density", fontsize=10)
ax_kde.set_title("KDE of Latency Distributions (P95 dotted)", fontsize=11)
ax_kde.legend(fontsize=9)

# --- Box plot ---
ax_box = axes[1]
bp_data   = [warm_cache, preloaded, cold_start]
bp_labels = ["Warm Cache", "Preloaded", "Cold Start"]
bp_colors = [COLORS["Warm Cache"], COLORS["Preloaded (Lifespan)"], COLORS["Cold Start (Download)"]]

bp = ax_box.boxplot(bp_data, vert=True, patch_artist=True,
                    showfliers=True, flierprops=dict(marker=".", markersize=2, alpha=0.4),
                    medianprops=dict(color="#f0e68c", lw=2.5),
                    whiskerprops=dict(lw=1.4),
                    capprops=dict(lw=1.4))
for patch, col in zip(bp["boxes"], bp_colors):
    patch.set_facecolor(col)
    patch.set_alpha(0.65)
for flier_col, col in zip(bp["fliers"], bp_colors):
    flier_col.set_markerfacecolor(col)

ax_box.set_yscale("log")
ax_box.set_xticklabels(bp_labels, fontsize=9)
ax_box.set_ylabel("Latency (ms) — log scale", fontsize=10)
ax_box.set_title("Box Plot with Outlier Detection", fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "s2_latency_distributions.png"),
            bbox_inches="tight", dpi=150, facecolor=fig.get_facecolor())
plt.show()
print("Charts saved to outputs/s2_latency_distributions.png")

---
## Section 3 — Token Distribution Analysis

**Background:** DeBERTa-v3-base has a 512-token context window. The NLI worker pipeline tokenizes the `(hypothesis, premise)` pair. Pairs that exceed 510 tokens (2-token overhead for `[CLS]` / `[SEP]`) require a **dual-pass** strategy: split the premise, score each chunk independently, and aggregate logits. This adds latency proportional to `ceil(token_count / 510)`.

**RQ-3:** What fraction of real requests hit the dual-pass path, and what is the P95 latency uplift?

In [ ]:
# ===========================================================================
# Section 3: Token Distribution Analysis
# WHY: The 510-token boundary is not a clean cliff — it's a distribution
#      problem. Knowing the tail beyond 510 tokens lets us budget for the
#      dual-pass latency overhead in SLO calculations.
# ===========================================================================

TOKEN_MEAN   = 380
TOKEN_STD    = 180
TOKEN_MAX    = 1024
TOKEN_MIN    = 8      # minimum meaningful pair
SPLIT_BOUNDARY = 510  # tokens above this require dual-pass
N_TOKEN_SAMPLES = 20_000

# Truncated normal (clip to [TOKEN_MIN, TOKEN_MAX])
a_clip = (TOKEN_MIN - TOKEN_MEAN) / TOKEN_STD
b_clip = (TOKEN_MAX - TOKEN_MEAN) / TOKEN_STD
token_dist = stats.truncnorm(a=a_clip, b=b_clip,
                              loc=TOKEN_MEAN, scale=TOKEN_STD)
token_counts = token_dist.rvs(size=N_TOKEN_SAMPLES,
                               random_state=RNG_SEED).astype(int)
token_counts = np.clip(token_counts, TOKEN_MIN, TOKEN_MAX)

single_pass_mask = token_counts <= SPLIT_BOUNDARY
dual_pass_mask   = ~single_pass_mask

pct_single = single_pass_mask.mean() * 100
pct_dual   = dual_pass_mask.mean()   * 100

# Latency impact: dual-pass adds ~1× warm_cache P50 latency per extra chunk
WARM_P50_MS = np.percentile(warm_cache, 50)
n_passes = np.ceil(token_counts / SPLIT_BOUNDARY).astype(int)
effective_latency = WARM_P50_MS * n_passes  # simplified linear model

p95_effective = np.percentile(effective_latency, 95)
p95_single    = WARM_P50_MS  # 100% single-pass baseline
uplift_pct    = (p95_effective - p95_single) / p95_single * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Token Distribution Analysis — Dual-Pass Boundary at 510 Tokens",
             fontsize=14, fontweight="bold", color="#c9d1d9")

# --- Histogram ---
ax_hist = axes[0]
bins = np.linspace(TOKEN_MIN, TOKEN_MAX, 80)
ax_hist.hist(token_counts[single_pass_mask], bins=bins,
             color=PALETTE_BLUE, alpha=0.78, label=f"Single-pass ({pct_single:.1f}%)",
             density=True, edgecolor="none")
ax_hist.hist(token_counts[dual_pass_mask], bins=bins,
             color=PALETTE_RED, alpha=0.78, label=f"Dual-pass ({pct_dual:.1f}%)",
             density=True, edgecolor="none")
ax_hist.axvline(SPLIT_BOUNDARY, color="#f0e68c", lw=2.2, linestyle="--",
                label=f"Split boundary ({SPLIT_BOUNDARY} tokens)")
ax_hist.set_xlabel("Token count (hypothesis + premise)", fontsize=10)
ax_hist.set_ylabel("Density", fontsize=10)
ax_hist.set_title("Token Count Distribution", fontsize=11)
ax_hist.legend(fontsize=9)

# --- CDF ---
ax_cdf = axes[1]
sorted_tokens = np.sort(token_counts)
cdf = np.arange(1, len(sorted_tokens) + 1) / len(sorted_tokens)
ax_cdf.plot(sorted_tokens, cdf * 100, color=PALETTE_BLUE, lw=2.4)
ax_cdf.axvline(SPLIT_BOUNDARY, color="#f0e68c", lw=2, linestyle="--",
               label=f"510-token boundary")
# Mark the CDF value at boundary
cdf_at_boundary = (token_counts <= SPLIT_BOUNDARY).mean() * 100
ax_cdf.axhline(cdf_at_boundary, color="#3fb950", lw=1.5, linestyle=":")
ax_cdf.scatter([SPLIT_BOUNDARY], [cdf_at_boundary], color="#f0e68c", s=80, zorder=5)
ax_cdf.text(SPLIT_BOUNDARY + 15, cdf_at_boundary - 4,
            f"{cdf_at_boundary:.1f}% ≤ 510 tokens",
            color="#3fb950", fontsize=9, fontweight="bold")
ax_cdf.set_xlabel("Token count", fontsize=10)
ax_cdf.set_ylabel("Cumulative % of requests", fontsize=10)
ax_cdf.set_title("CDF of Token Counts", fontsize=11)
ax_cdf.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "s3_token_distribution.png"),
            bbox_inches="tight", dpi=150, facecolor=fig.get_facecolor())
plt.show()

print("\n=== Token Distribution Summary ===")
print(f"  Total samples      : {N_TOKEN_SAMPLES:,}")
print(f"  Mean token count   : {token_counts.mean():.1f}")
print(f"  Std token count    : {token_counts.std():.1f}")
print(f"  Single-pass (≤510) : {pct_single:.2f}%")
print(f"  Dual-pass   (>510) : {pct_dual:.2f}%")
print(f"  Warm P50 latency   : {WARM_P50_MS:.1f} ms")
print(f"  P95 effective latency (with dual-pass) : {p95_effective:.1f} ms")
print(f"  P95 latency uplift due to dual-pass    : +{uplift_pct:.2f}%")

---
## Section 4 — Algorithm Complexity Benchmarking

The ADR mandates 17 specific algorithms. We benchmark a representative subset spanning O(1) → O(n log n). Each benchmark answers: *does the empirical timing confirm the theoretical complexity class?*

> **Testing methodology:** Each algorithm is run across a range of input sizes N. We fit a power-law `t ∝ N^k` and compare `k` to the theoretical exponent. Agreement within ±0.15 is accepted as confirmation.

In [ ]:
# ===========================================================================
# Section 4a: ALG-02 — EWMA Convergence Speed (α sensitivity)
# WHY: The smoothing factor α is the single most important tuning knob.
#      High α = fast tracking but noisy; low α = smooth but laggy.
#      We need empirical evidence for the α=0.3 default in the ADR.
# ===========================================================================

N_EWMA = 200
# Synthetic signal: step change at t=100 (simulates a latency spike event)
signal = np.concatenate([rng.normal(30, 3, 100), rng.normal(80, 5, 100)])
alphas = [0.1, 0.3, 0.5]
EWMA_COLORS = ["#388bfd", "#3fb950", "#f85149"]

def compute_ewma(signal: np.ndarray, alpha: float) -> np.ndarray:
    out = np.empty_like(signal)
    out[0] = signal[0]
    for i in range(1, len(signal)):
        out[i] = alpha * signal[i] + (1 - alpha) * out[i - 1]
    return out

# Measure convergence: samples needed to reach 90% of step change
STEP_CHANGE = 80 - 30  # = 50 ms
THRESH = 30 + 0.9 * STEP_CHANGE  # = 75 ms

fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle("ALG-02: EWMA Convergence Speed — α Sensitivity Analysis",
             fontsize=13, fontweight="bold", color="#c9d1d9")

ax.plot(signal, color="#8b949e", lw=1.2, alpha=0.65, label="Raw signal", zorder=1)
ax.axvline(100, color="#f0e68c", lw=1.5, linestyle="--", alpha=0.7, label="Step change (t=100)")

convergence_report = []
for alpha, col in zip(alphas, EWMA_COLORS):
    ewma = compute_ewma(signal, alpha)
    ax.plot(ewma, color=col, lw=2.2, label=f"α={alpha}")
    # Find convergence point (first index after step where EWMA >= 75)
    post_step = ewma[100:]
    convergence_idx = np.argmax(post_step >= THRESH)
    if post_step[convergence_idx] < THRESH:
        convergence_idx = len(post_step)  # never converged
    convergence_report.append((alpha, convergence_idx))
    if convergence_idx < len(post_step):
        ax.scatter([100 + convergence_idx], [ewma[100 + convergence_idx]],
                   color=col, s=70, zorder=5)

ax.set_xlabel("Sample index (each = 1 request)", fontsize=10)
ax.set_ylabel("EWMA value (ms)", fontsize=10)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "s4a_ewma_convergence.png"),
            bbox_inches="tight", dpi=150, facecolor=fig.get_facecolor())
plt.show()

print("=== ALG-02 EWMA Convergence (samples to reach 90% of step change) ===")
for alpha, n_samples in convergence_report:
    comment = " ← ADR default" if alpha == 0.3 else ""
    print(f"  α={alpha}: {n_samples:>4} samples to 90% convergence{comment}")
print()

In [ ]:
# ===========================================================================
# Section 4b: ALG-03 — Bloom Filter False Positive Rate vs Capacity
# WHY: The ADR uses a Bloom filter for request deduplication in the
#      trace ingestion path. We must verify that with k=3 hash functions
#      and n items, the FPR stays below 1% before we hit capacity.
# ===========================================================================

K_HASH = 3  # number of hash functions

def bloom_fpr_theoretical(n: int, m: int, k: int) -> float:
    """
    Theoretical FPR = (1 - e^(-k*n/m))^k
    n = items inserted, m = bit-array size, k = hash functions
    """
    return (1 - np.exp(-k * n / m)) ** k

def empirical_bloom_fpr(n: int, m: int, k: int, n_test: int = 10_000) -> float:
    """Simulate a Bloom filter and measure empirical FPR."""
    bit_array = np.zeros(m, dtype=bool)
    inserted = set()

    def get_positions(item: str):
        positions = []
        for seed in range(k):
            h = int(hashlib.md5(f"{seed}:{item}".encode()).hexdigest(), 16)
            positions.append(h % m)
        return positions

    # Insert n items
    for i in range(n):
        item = f"trace-id-{i}"
        inserted.add(item)
        for pos in get_positions(item):
            bit_array[pos] = True

    # Test n_test truly-absent items
    fp = 0
    for i in range(n_test):
        probe = f"probe-{n + i}"
        if probe not in inserted:
            if all(bit_array[pos] for pos in get_positions(probe)):
                fp += 1
    return fp / n_test

M_FILTER = 50_000  # bit-array size (≈6.1 KB)
n_range = np.arange(1000, 16001, 1000)
theoretical_fpr = [bloom_fpr_theoretical(n, M_FILTER, K_HASH) * 100 for n in n_range]

# Empirical at a few key points (fast: m=50k is small)
empirical_points_n = [2000, 5000, 8000, 12000]
empirical_fpr = [empirical_bloom_fpr(n, M_FILTER, K_HASH, n_test=5000) * 100
                 for n in empirical_points_n]

fig, ax = plt.subplots(figsize=(11, 5))
fig.suptitle(f"ALG-03: Bloom Filter FPR vs Items Inserted  (m={M_FILTER:,} bits, k={K_HASH})",
             fontsize=13, fontweight="bold", color="#c9d1d9")

ax.plot(n_range, theoretical_fpr, color=PALETTE_BLUE, lw=2.4, label="Theoretical FPR")
ax.scatter(empirical_points_n, empirical_fpr, color=PALETTE_RED, s=80, zorder=5,
           label="Empirical FPR (simulation)", edgecolors="#c9d1d9", linewidths=0.8)
ax.axhline(1.0, color="#f0e68c", lw=1.8, linestyle="--", label="1% FPR threshold")

# Find capacity at 1% FPR
cap_idx = next((i for i, v in enumerate(theoretical_fpr) if v > 1.0), None)
if cap_idx:
    ax.axvline(n_range[cap_idx], color="#f85149", lw=1.5, linestyle=":",
               label=f"Capacity limit (~{n_range[cap_idx]:,} items)")

ax.set_xlabel("Items inserted (n)", fontsize=10)
ax.set_ylabel("False Positive Rate (%)", fontsize=10)
ax.legend(fontsize=9)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "s4b_bloom_filter_fpr.png"),
            bbox_inches="tight", dpi=150, facecolor=fig.get_facecolor())
plt.show()

print("\n=== ALG-03 Bloom Filter Analysis ===")
print(f"  Bit-array size m : {M_FILTER:,} bits ({M_FILTER/8/1024:.2f} KB)")
print(f"  Hash functions k : {K_HASH}")
print(f"  Optimal item cap : ~{int(M_FILTER * (math.log(2)**2 / K_HASH)):,} items at {K_HASH} hashes")
for n, emp in zip(empirical_points_n, empirical_fpr):
    th = bloom_fpr_theoretical(n, M_FILTER, K_HASH) * 100
    print(f"  n={n:>6,}: theory={th:.3f}%  empirical={emp:.3f}%  diff={abs(emp-th):.3f}%")
print()

In [ ]:
# ===========================================================================
# Section 4c: ALG-06 — Min-Heap O(log n) vs Sorted List O(n) Timing
# WHY: The ADR uses a min-heap for the top-K toxic span tracker.
#      Empirically verifying the O(log n) vs O(n) gap is essential to
#      justify the choice at high throughput (n > 10K concurrent spans).
# ===========================================================================

    "N_SIZES = [100, 500, 1000, 2000, 5000, 10000]
",
    "REPS    = 10
",

heap_times    = []
sortedlist_times = []

    "N_SIZES = [100, 500, 1000, 2000, 5000, 10000]
",
    data = rng.integers(0, 1_000_000, size=n).tolist()

    # Heap insertion benchmark
    t0 = time.perf_counter()
    for _ in range(REPS):
        h = []
        for x in data:
            heapq.heappush(h, x)
    heap_times.append((time.perf_counter() - t0) / REPS * 1e6)  # µs

    # Sorted list insertion benchmark (O(n) per insert)
    t0 = time.perf_counter()
    for _ in range(REPS):
        sl = []
        for x in data:
            # bisect.insort is O(log n) search + O(n) insert (memmove)
            import bisect
            bisect.insort(sl, x)
    sortedlist_times.append((time.perf_counter() - t0) / REPS * 1e6)

# Fit power-law: log(t) = k*log(n) + c
def fit_exponent(sizes, times):
    log_n = np.log(sizes)
    log_t = np.log(times)
    k, c = np.polyfit(log_n, log_t, 1)
    return k, c

    "N_SIZES = [100, 500, 1000, 2000, 5000, 10000]
",
    "N_SIZES = [100, 500, 1000, 2000, 5000, 10000]
",

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle("ALG-06: Min-Heap O(log n) vs Sorted List O(n) — Empirical Timing",
             fontsize=13, fontweight="bold", color="#c9d1d9")

    "N_SIZES = [100, 500, 1000, 2000, 5000, 10000]
",
          label=f"heapq (empirical k={heap_k:.2f}, theory=1.00)")
    "N_SIZES = [100, 500, 1000, 2000, 5000, 10000]
",
          label=f"sorted list (empirical k={sort_k:.2f}, theory=2.00)")

ax.set_xlabel("Input size n", fontsize=10)
ax.set_ylabel("Total insertion time (µs)", fontsize=10)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "s4c_heap_vs_sortedlist.png"),
            bbox_inches="tight", dpi=150, facecolor=fig.get_facecolor())
plt.show()

print("=== ALG-06 Complexity Verification ===")
print(f"  Min-heap power-law exponent    : k={heap_k:.3f}  (theoretical: 1.00 for O(n log n) total)")
print(f"  Sorted-list power-law exponent : k={sort_k:.3f}  (theoretical: 2.00 for O(n²) total)")
print(f"  Agreement within ±0.15        : heap={'YES' if abs(heap_k-1.0)<=0.15 else 'NO'}  sorted={'YES' if abs(sort_k-2.0)<=0.15 else 'NO'}")
print(f"  Speed ratio at n=10K          : {sortedlist_times[-1]/heap_times[-1]:.1f}× faster with heap")
print()

In [ ]:
# ===========================================================================
# Section 4d: ALG-10 — Count-Min Sketch Error Rate vs Table Width
# WHY: CMS is used for approximate frequency counting of trace IDs.
#      The error guarantee is ε = e/w with probability 1 - δ = 1 - e^(-d)
#      where w = width, d = depth. We verify empirically.
# ===========================================================================

DEPTHS  = [3, 4, 5]    # number of hash functions (rows)
WIDTHS  = [500, 1000, 2000, 3000, 5000, 8000]
    "N_CMS_ITEMS = 1000
",
    "CMS_REPS    = 2
",

class CountMinSketch:
    def __init__(self, w: int, d: int):
        self.w = w
        self.d = d
        self.table = [[0] * w for _ in range(d)]

    def _hashes(self, item: str):
        for i in range(self.d):
            h = int(hashlib.md5(f"{i}:{item}".encode()).hexdigest(), 16)
            yield h % self.w

    def add(self, item: str):
        for i, col in enumerate(self._hashes(item)):
            self.table[i][col] += 1

    def query(self, item: str) -> int:
        return min(self.table[i][col]
                   for i, col in enumerate(self._hashes(item)))

# Generate skewed Zipf-like true counts
    "N_CMS_ITEMS = 1000
",
# Zipf distribution: item i has frequency ∝ 1/i
    "N_CMS_ITEMS = 1000
",
weights /= weights.sum()
    "N_OPS = 20_000
",
    "N_OPS = 20_000
",
true_counts = Counter(sampled_items)

cms_results = defaultdict(list)  # depth → list of (width, mean_rel_error)
for d in DEPTHS:
    for w in WIDTHS:
        errors = []
    "CMS_REPS    = 2
",
            cms = CountMinSketch(w=w, d=d)
            for item in sampled_items:
                cms.add(item)
            # Relative error on top-100 items
            for item in list(true_counts.keys())[:100]:
                true_c = true_counts[item]
                est_c  = cms.query(item)
                errors.append(abs(est_c - true_c) / max(true_c, 1))
        cms_results[d].append(np.mean(errors) * 100)

fig, ax = plt.subplots(figsize=(11, 5))
fig.suptitle("ALG-10: Count-Min Sketch — Mean Relative Error vs Table Width",
             fontsize=13, fontweight="bold", color="#c9d1d9")

depth_colors = [PALETTE_RED, PALETTE_BLUE, "#3fb950"]
for d, col in zip(DEPTHS, depth_colors):
    eps_theory = [math.e / w * 100 for w in WIDTHS]  # theoretical ε = e/w
    ax.plot(WIDTHS, cms_results[d], "o-", color=col, lw=2.2,
            label=f"d={d} (empirical)")
ax.plot(WIDTHS, [math.e / w * 100 for w in WIDTHS], "k--",
        lw=1.8, alpha=0.7, label="Theoretical ε = e/w (d=any)")
ax.axhline(1.0, color="#f0e68c", lw=1.5, linestyle=":", label="1% error threshold")
ax.set_xlabel("Table width w", fontsize=10)
ax.set_ylabel("Mean relative error (%)", fontsize=10)
ax.legend(fontsize=9)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "s4d_count_min_sketch.png"),
            bbox_inches="tight", dpi=150, facecolor=fig.get_facecolor())
plt.show()

print("\n=== ALG-10 Count-Min Sketch Error Rates ===")
for w, err_d3 in zip(WIDTHS, cms_results[3]):
    theory = math.e / w * 100
    print(f"  w={w:>5,}: empirical={err_d3:.3f}%  theory={theory:.3f}%")
print()

In [ ]:
# ===========================================================================
# Section 4e: ALG-14 Wilson Score Interval + ALG-15 Welch's Power Curve
# WHY: Wilson intervals are used for SLO compliance confidence bounds.
#      The power curve justifies the minimum sample size needed to detect
#      a 10% shift in mean toxicity score with 80% power.
# ===========================================================================

# --- ALG-14: Wilson Score CI width vs sample size ---
P_HAT  = 0.95   # assumed compliance rate (point estimate)
Z_95   = 1.96

sample_sizes = np.arange(10, 2001, 10)

def wilson_interval(n: int, p_hat: float, z: float):
    """Wilson score interval for binomial proportion."""
    center = (p_hat + z**2 / (2*n)) / (1 + z**2 / n)
    half_w = z * np.sqrt(p_hat*(1-p_hat)/n + z**2/(4*n**2)) / (1 + z**2/n)
    return center - half_w, center + half_w

ci_widths = []
for n in sample_sizes:
    lo, hi = wilson_interval(n, P_HAT, Z_95)
    ci_widths.append((hi - lo) * 100)  # percentage points

# --- ALG-15: Statistical power for detecting 10% toxicity score shift ---
MU0   = 0.50  # baseline mean toxicity score
MU1   = 0.55  # alternative (10% relative shift)
SIGMA = 0.12  # assumed std dev
ALPHA = 0.05

power_sizes = np.arange(5, 501, 5)

def compute_power(n: int, mu0, mu1, sigma, alpha) -> float:
    """Power of two-sample Welch's t-test (equal n, equal sigma)."""
    delta = abs(mu1 - mu0)
    se    = sigma * np.sqrt(2 / n)
    t_crit = stats.t.ppf(1 - alpha/2, df=2*(n-1))
    ncp = delta / se  # non-centrality parameter
    power = 1 - stats.t.cdf(t_crit, df=2*(n-1), loc=ncp)
    return power

powers = [compute_power(n, MU0, MU1, SIGMA, ALPHA) for n in power_sizes]
min_n_80 = power_sizes[next(i for i, p in enumerate(powers) if p >= 0.80)]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("ALG-14 Wilson CI Width · ALG-15 Welch's Power Curve",
             fontsize=13, fontweight="bold", color="#c9d1d9")

# Wilson plot
ax_w = axes[0]
ax_w.plot(sample_sizes, ci_widths, color=PALETTE_BLUE, lw=2.2)
ax_w.axhline(1.0, color="#f0e68c", linestyle="--", lw=1.5,
             label="1 ppt precision threshold")
n_1ppt = sample_sizes[np.argmax(np.array(ci_widths) <= 1.0)]
ax_w.axvline(n_1ppt, color="#3fb950", lw=1.5, linestyle=":")
ax_w.text(n_1ppt + 20, 2.0, f"n={n_1ppt}\nfor <1 ppt",
          color="#3fb950", fontsize=9, fontweight="bold")
ax_w.set_xlabel("Sample size n", fontsize=10)
ax_w.set_ylabel("Wilson CI width (percentage points)", fontsize=10)
ax_w.set_title("ALG-14: Wilson Score CI Width vs Sample Size", fontsize=11)
ax_w.legend(fontsize=9)

# Power curve
ax_p = axes[1]
ax_p.plot(power_sizes, powers, color=PALETTE_RED, lw=2.2,
          label=f"δ=10% shift (μ={MU0}→{MU1}, σ={SIGMA})")
ax_p.axhline(0.80, color="#f0e68c", linestyle="--", lw=1.5, label="80% power threshold")
ax_p.axvline(min_n_80, color="#3fb950", lw=1.5, linestyle=":")
ax_p.text(min_n_80 + 5, 0.55, f"n={min_n_80}\nper group",
          color="#3fb950", fontsize=9, fontweight="bold")
ax_p.set_xlabel("Sample size per group", fontsize=10)
ax_p.set_ylabel("Statistical power", fontsize=10)
ax_p.set_title("ALG-15: Power Curve for 10% Toxicity Score Shift", fontsize=11)
ax_p.legend(fontsize=9)
ax_p.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "s4e_wilson_power.png"),
            bbox_inches="tight", dpi=150, facecolor=fig.get_facecolor())
plt.show()

print("=== ALG-14 Wilson Score ===")
print(f"  At n=100:  CI = [{wilson_interval(100,  P_HAT, Z_95)[0]*100:.2f}%, {wilson_interval(100,  P_HAT, Z_95)[1]*100:.2f}%]")
print(f"  At n=500:  CI = [{wilson_interval(500,  P_HAT, Z_95)[0]*100:.2f}%, {wilson_interval(500,  P_HAT, Z_95)[1]*100:.2f}%]")
print(f"  At n=1000: CI = [{wilson_interval(1000, P_HAT, Z_95)[0]*100:.2f}%, {wilson_interval(1000, P_HAT, Z_95)[1]*100:.2f}%]")
print(f"  Min n for <1 ppt CI width: {n_1ppt}")
print(f"\n=== ALG-15 Welch's Power ===")
print(f"  Min n per group for 80% power (δ=10% shift): {min_n_80}")
print(f"  Power at n=50 : {compute_power(50,  MU0, MU1, SIGMA, ALPHA)*100:.1f}%")
print(f"  Power at n=100: {compute_power(100, MU0, MU1, SIGMA, ALPHA)*100:.1f}%")
print()

---
## Section 5 — Concurrency Simulation (Double-Checked Locking)

**Problem:** The model registry uses a global `threading.Lock` for the registry dict and per-model `threading.Lock` objects for download serialization. Without double-checked locking, every fast-path cache hit would needlessly contend on the global lock.

**Simulation design:** We launch 20 threads. 15 threads request cached models (fast path). 5 threads request new models (slow path). We measure actual wall-clock contention using `time.sleep()` to model realistic download latency.

In [ ]:
# ===========================================================================
# Section 5: Concurrency Simulation with Actual Threading
# WHY: Simulating locking with print statements (old notebook) proves nothing.
#      Actual threading.Lock() + time.sleep() reveals real contention
#      patterns and confirms that per-model locks do NOT block each other.
# ===========================================================================

N_THREADS     = 20
CACHED_MODEL_LATENCY_S  = 0.026   # 26 ms (warm cache)
DOWNLOAD_LATENCY_S      = 0.200   # 200 ms (scaled down from 7.2s for notebook speed)
DOWNLOAD_THREAD_IDS     = {2, 4, 7, 12, 17}  # threads that request new models

# Shared state
registry = {}                  # model_name → model_object
registry_lock  = threading.Lock()
model_locks    = {}            # per-model lock dict
model_locks_mu = threading.Lock()

# Telemetry
thread_events = []  # list of (thread_id, label, start_time, end_time)
events_lock   = threading.Lock()

T_ORIGIN = time.perf_counter()

def get_model(thread_id: int, model_name: str):
    t_enter = time.perf_counter() - T_ORIGIN

    # --- FAST PATH: double-check (no lock) ---
    if model_name in registry:
        time.sleep(CACHED_MODEL_LATENCY_S)
        t_exit = time.perf_counter() - T_ORIGIN
        with events_lock:
            thread_events.append((thread_id, "CACHE HIT", t_enter, t_exit))
        return registry[model_name]

    # --- Per-model lock acquisition ---
    with model_locks_mu:
        if model_name not in model_locks:
            model_locks[model_name] = threading.Lock()
    per_model_lock = model_locks[model_name]

    t_lock_wait_start = time.perf_counter() - T_ORIGIN
    with per_model_lock:
        t_lock_acquired = time.perf_counter() - T_ORIGIN
        # Second check inside lock
        if model_name not in registry:
            with events_lock:
                thread_events.append((thread_id, "DOWNLOAD", t_lock_wait_start, None))
            time.sleep(DOWNLOAD_LATENCY_S)   # simulate download
            with registry_lock:
                registry[model_name] = object()  # placeholder model
            t_exit = time.perf_counter() - T_ORIGIN
            with events_lock:
                # Update the None sentinel
                for e in thread_events:
                    if e[0] == thread_id and e[1] == "DOWNLOAD" and e[3] is None:
                        idx = thread_events.index(e)
                        thread_events[idx] = (thread_id, "DOWNLOAD", t_lock_wait_start, t_exit)
                        break
        else:
            # Race loser: model was downloaded by peer
            time.sleep(CACHED_MODEL_LATENCY_S)
            t_exit = time.perf_counter() - T_ORIGIN
            with events_lock:
                thread_events.append((thread_id, "LOCK WAIT HIT", t_lock_wait_start, t_exit))

threads = []
for i in range(N_THREADS):
    if i in DOWNLOAD_THREAD_IDS:
        model_name = f"model-{'ab'[list(DOWNLOAD_THREAD_IDS).index(i) % 2]}"
    else:
        model_name = "cached-model"
        registry["cached-model"] = object()  # pre-populate cache
    t = threading.Thread(target=get_model, args=(i, model_name), daemon=True)
    threads.append(t)

for t in threads:
    t.start()
for t in threads:
    t.join(timeout=5.0)

print(f"All {N_THREADS} threads completed.")
print(f"Registry now contains: {list(registry.keys())}")
print()

# --- Compute statistics ---
wait_times    = []
event_summary = Counter(e[1] for e in thread_events)
for e in thread_events:
    if e[3] is not None:
        wait_ms = (e[3] - e[2]) * 1000
        wait_times.append((e[0], e[1], wait_ms))

print("=== Thread Event Summary ===")
for label, cnt in sorted(event_summary.items()):
    print(f"  {label:<18}: {cnt} threads")

if wait_times:
    all_wait_ms = [w[2] for w in wait_times]
    print(f"\n  Max wait time   : {max(all_wait_ms):.1f} ms")
    print(f"  Avg wait time   : {np.mean(all_wait_ms):.1f} ms")
    cache_hits = [w for w in wait_times if w[1] == "CACHE HIT"]
    downloads  = [w for w in wait_times if w[1] in ("DOWNLOAD", "LOCK WAIT HIT")]
    print(f"  Lock contention : {len(downloads)}/{N_THREADS} threads blocked on model lock")
    print(f"  Contention rate : {len(downloads)/N_THREADS*100:.1f}%")

In [ ]:
# ===========================================================================
# Section 5b: Gantt Chart of Thread Execution Timelines
# WHY: A Gantt chart makes the concurrency pattern immediately legible:
#      cache hits complete in <<1 bar width; download threads are the
#      long bars; per-model locks do NOT serialize across different models.
# ===========================================================================

LABEL_COLORS = {
    "CACHE HIT":    PALETTE_BLUE,
    "DOWNLOAD":     PALETTE_RED,
    "LOCK WAIT HIT": "#d29922",
}

valid_events = [(tid, lbl, s, e) for tid, lbl, s, e in thread_events if e is not None]
valid_events.sort(key=lambda x: x[0])

fig, ax = plt.subplots(figsize=(14, 8))
fig.suptitle("Concurrency Gantt Chart — Thread Execution Timelines",
             fontsize=13, fontweight="bold", color="#c9d1d9")

y_positions = {}
y_tick_labels = []

for row_idx, (tid, lbl, start, end) in enumerate(valid_events):
    y = tid
    duration = (end - start) * 1000  # ms
    ax.barh(y, duration, left=start * 1000,
            height=0.55,
            color=LABEL_COLORS.get(lbl, "#8b949e"),
            alpha=0.82, edgecolor="#30363d", zorder=3)
    if duration > 8:
        ax.text(start * 1000 + duration / 2, y,
                f"{lbl}\n{duration:.0f} ms",
                ha="center", va="center", fontsize=6.5,
                color="#0d1117", fontweight="bold")

# Legend
legend_patches = [mpatches.Patch(color=c, label=l)
                  for l, c in LABEL_COLORS.items()]
ax.legend(handles=legend_patches, loc="upper right", fontsize=9)

ax.set_xlabel("Wall-clock time (ms)", fontsize=10)
ax.set_ylabel("Thread ID", fontsize=10)
ax.set_yticks(range(N_THREADS))
ax.set_yticklabels([f"T-{i}" for i in range(N_THREADS)], fontsize=7)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "s5_concurrency_gantt.png"),
            bbox_inches="tight", dpi=150, facecolor=fig.get_facecolor())
plt.show()
print("Gantt chart saved to outputs/s5_concurrency_gantt.png")

---
## Section 6 — Circuit Breaker State Machine Simulation

**Background:** The NLI worker calls an upstream inference server. If that server is degraded (e.g., GPU OOM), unchecked retries cascade load. The circuit breaker FSM prevents this:

```
CLOSED ──(failure_rate > threshold)──► OPEN
  ▲                                      │
  │          (reset_timeout elapsed)     │
  │      ◄────────────────────────────── │
CLOSED ◄─(probe succeeds)── HALF-OPEN ◄─┘
                    │
                    └──(probe fails)──► OPEN
```

**Parameters:**
- Failure threshold: 50% failures in last 10 requests → trip to OPEN
- Reset timeout: 15 requests (simulated)
- Half-open probe: 1 request; success → CLOSED, failure → OPEN

In [ ]:
# ===========================================================================
# Section 6: Circuit Breaker FSM Simulation
# WHY: The circuit breaker is the primary cascade-failure prevention
#      mechanism. We must prove it trips correctly at the design failure
#      rate (15%) and that recovery time matches the configured window.
# ===========================================================================

N_REQUESTS       = 120
FAILURE_RATE     = 0.18   # 18% upstream failure rate
FAIL_WINDOW      = 10     # rolling window for failure rate calculation
TRIP_THRESHOLD   = 0.50   # trip if >=50% failures in window
RESET_TIMEOUT    = 15     # requests to wait in OPEN before moving to HALF-OPEN

STATES = ["CLOSED", "OPEN", "HALF-OPEN"]
STATE_COLORS = {"CLOSED": PALETTE_BLUE, "OPEN": PALETTE_RED, "HALF-OPEN": "#d29922"}
STATE_Y      = {"CLOSED": 0, "HALF-OPEN": 1, "OPEN": 2}

# Reproducible failure mask
fail_oracle = rng.random(N_REQUESTS) < FAILURE_RATE

state = "CLOSED"
open_since = None
history = []            # (request_id, state, was_allowed, was_failure)
recent_outcomes = []    # rolling window
avoided_failures = 0
state_transitions = []

for req_id in range(N_REQUESTS):
    allowed = False
    outcome_fail = False

    if state == "CLOSED":
        allowed = True
        outcome_fail = bool(fail_oracle[req_id])
        recent_outcomes.append(1 if outcome_fail else 0)
        if len(recent_outcomes) > FAIL_WINDOW:
            recent_outcomes.pop(0)
        # Trip check
        if (len(recent_outcomes) >= FAIL_WINDOW
                and np.mean(recent_outcomes) >= TRIP_THRESHOLD):
            prev_state = state
            state = "OPEN"
            open_since = req_id
            state_transitions.append((req_id, "CLOSED→OPEN"))
            recent_outcomes.clear()

    elif state == "OPEN":
        allowed = False
        avoided_failures += 1 if bool(fail_oracle[req_id]) else 0
        if req_id - open_since >= RESET_TIMEOUT:
            state = "HALF-OPEN"
            state_transitions.append((req_id, "OPEN→HALF-OPEN"))

    elif state == "HALF-OPEN":
        allowed = True
        outcome_fail = bool(fail_oracle[req_id])
        if outcome_fail:
            state = "OPEN"
            open_since = req_id
            state_transitions.append((req_id, "HALF-OPEN→OPEN"))
        else:
            state = "CLOSED"
            recent_outcomes.clear()
            state_transitions.append((req_id, "HALF-OPEN→CLOSED"))

    history.append((req_id, state, allowed, outcome_fail))

# Compute rolling failure rate (for CLOSED-state requests only)
df_hist = pd.DataFrame(history, columns=["req_id", "state", "allowed", "failed"])
df_hist["state_y"] = df_hist["state"].map(STATE_Y)
df_hist["rolling_fail"] = (
    df_hist["failed"].rolling(FAIL_WINDOW, min_periods=1).mean() * 100
)

# --- Metrics ---
open_durations = []
in_open_since  = None
for tr in state_transitions:
    r, label = tr
    if "→OPEN" in label:
        in_open_since = r
    elif in_open_since is not None and ("OPEN→" in label):
        open_durations.append(r - in_open_since)
        in_open_since = None

mttr = np.mean(open_durations) if open_durations else 0
n_trips = sum(1 for _, l in state_transitions if "→OPEN" in l)

# --- Plot ---
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle("Circuit Breaker FSM Simulation — 120 Requests at 18% Failure Rate",
             fontsize=14, fontweight="bold", color="#c9d1d9")

# Plot 1: State over time
ax_state = axes[0]
for state_label, sy in STATE_Y.items():
    mask = df_hist["state"] == state_label
    ax_state.scatter(df_hist.loc[mask, "req_id"], [sy] * mask.sum(),
                     color=STATE_COLORS[state_label], s=28, zorder=3,
                     label=state_label)
for req_id, label in state_transitions:
    ax_state.axvline(req_id, color="#8b949e", lw=1.0, linestyle=":", alpha=0.6)
    ax_state.text(req_id, 2.4, label.split("→")[1][:2],
                  fontsize=6, color="#8b949e", ha="center")
ax_state.set_yticks([0, 1, 2])
ax_state.set_yticklabels(["CLOSED", "HALF-OPEN", "OPEN"], fontsize=9)
ax_state.set_ylabel("CB State", fontsize=9)
ax_state.legend(loc="upper right", fontsize=8)
ax_state.set_title("Circuit Breaker State over Request Sequence", fontsize=11)

# Plot 2: Rolling failure rate
ax_fail = axes[1]
ax_fail.plot(df_hist["req_id"], df_hist["rolling_fail"],
             color=PALETTE_RED, lw=2.0, label=f"Rolling failure rate ({FAIL_WINDOW}-req window)")
ax_fail.axhline(TRIP_THRESHOLD * 100, color="#f0e68c", lw=1.8, linestyle="--",
                label=f"Trip threshold ({TRIP_THRESHOLD*100:.0f}%)")
ax_fail.set_ylabel("Failure rate (%)", fontsize=9)
ax_fail.legend(fontsize=8)
ax_fail.set_ylim(0, 100)

# Plot 3: Allowed vs blocked requests
ax_allow = axes[2]
allowed_mask = df_hist["allowed"]
blocked_mask = ~df_hist["allowed"]
ax_allow.scatter(df_hist.loc[allowed_mask, "req_id"],
                 [1] * allowed_mask.sum(),
                 color=PALETTE_BLUE, s=25, alpha=0.8, label="Allowed")
ax_allow.scatter(df_hist.loc[blocked_mask, "req_id"],
                 [0] * blocked_mask.sum(),
                 color=PALETTE_RED, s=25, alpha=0.8, label="Blocked (OPEN)")
ax_allow.set_yticks([0, 1])
ax_allow.set_yticklabels(["Blocked", "Allowed"], fontsize=9)
ax_allow.set_xlabel("Request sequence number", fontsize=10)
ax_allow.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "s6_circuit_breaker.png"),
            bbox_inches="tight", dpi=150, facecolor=fig.get_facecolor())
plt.show()

print("\n=== Circuit Breaker FSM Metrics ===")
print(f"  Total requests        : {N_REQUESTS}")
print(f"  Configured fail rate  : {FAILURE_RATE*100:.0f}%")
print(f"  Trip events           : {n_trips}")
print(f"  State transitions     : {len(state_transitions)}")
for r, label in state_transitions:
    print(f"    req #{r:>3}: {label}")
print(f"  Mean time to recovery : {mttr:.1f} requests")
print(f"  Requests blocked      : {blocked_mask.sum()} ({blocked_mask.sum()/N_REQUESTS*100:.1f}%)")
print(f"  Cascade failures avoided (OPEN-state failures shunted): {avoided_failures}")

---
## Section 7 — Memory Consumption Analysis

**Model RAM footprints:**

| Model | Format | RAM |
|---|---|---|
| `cross-encoder/nli-deberta-v3-base` | PyTorch FP32 | ~1,400 MB |
| `martin-ha/toxic-comment-model` | ONNX INT8 quantized | ~440 MB |

With multiple `(tenant, model_variant)` pairs cached simultaneously, we risk OOM at the 8 GB container memory limit (default Kubernetes `resources.limits.memory`).

**Bloom filter sizing** for n=1M items at p=1% FPR:
$$m = -\frac{n \ln p}{(\ln 2)^2}$$

In [ ]:
# ===========================================================================
# Section 7: Memory Consumption Analysis
# WHY: The registry caches model objects in-process. Without an eviction
#      policy, each new (model, quantization) pair grows heap linearly.
#      We must identify the OOM cliff and size the LRU eviction cache.
# ===========================================================================

DEBERTA_MB   = 1400   # DeBERTa-v3-base PyTorch FP32
TOXIC_MB     =  440   # toxic-comment-model ONNX INT8
PYTHON_BASE_MB = 280  # Python runtime + FastAPI + torch skeleton (no weights)
CONTAINER_LIMIT_MB = 8 * 1024   # 8 GB Kubernetes limit
SAFETY_MARGIN_MB   = 512        # reserve for OS, tokenizer, request buffers
EFFECTIVE_LIMIT_MB = CONTAINER_LIMIT_MB - SAFETY_MARGIN_MB

# Model loading sequence: alternating DeBERTa / Toxic loads
max_pairs = 12
n_pairs   = np.arange(0, max_pairs + 1)

# Scenario A: Only DeBERTa models (multi-tenant, same architecture)
ram_deberta_only = PYTHON_BASE_MB + n_pairs * DEBERTA_MB
# Scenario B: Only Toxic models
ram_toxic_only   = PYTHON_BASE_MB + n_pairs * TOXIC_MB
# Scenario C: Alternating
ram_mixed = PYTHON_BASE_MB + np.array(
    [math.ceil(i/2) * DEBERTA_MB + math.floor(i/2) * TOXIC_MB
     for i in range(max_pairs + 1)]
)

# Bloom filter memory calculation
# m = -n * ln(p) / (ln(2))^2
BF_N = 1_000_000
BF_P = 0.01
bf_bits  = -BF_N * math.log(BF_P) / (math.log(2) ** 2)
bf_bytes = bf_bits / 8
bf_kb    = bf_bytes / 1024

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Memory Consumption Analysis — Model Registry Cache",
             fontsize=14, fontweight="bold", color="#c9d1d9")

# --- RAM curve ---
ax_ram = axes[0]
ax_ram.plot(n_pairs, ram_deberta_only / 1024, "o-", color=PALETTE_RED, lw=2.2,
            label="DeBERTa-only (1,400 MB each)")
ax_ram.plot(n_pairs, ram_toxic_only / 1024, "s-", color=PALETTE_BLUE, lw=2.2,
            label="Toxic-ONNX-only (440 MB each)")
ax_ram.plot(n_pairs, ram_mixed / 1024, "^--", color="#3fb950", lw=2.2,
            label="Mixed alternating")
ax_ram.axhline(CONTAINER_LIMIT_MB / 1024, color="#f85149", lw=2, linestyle="-",
               label=f"Container OOM limit ({CONTAINER_LIMIT_MB//1024} GB)")
ax_ram.axhline(EFFECTIVE_LIMIT_MB / 1024, color="#f0e68c", lw=1.8, linestyle="--",
               label=f"Effective limit w/ margin ({EFFECTIVE_LIMIT_MB//1024:.1f} GB)")

# Annotate OOM crossings
for ram_arr, name, col in [
    (ram_deberta_only, "DeBERTa-only", PALETTE_RED),
    (ram_mixed, "Mixed", "#3fb950"),
]:
    oom_idx = np.argmax(ram_arr > EFFECTIVE_LIMIT_MB)
    if ram_arr[oom_idx] > EFFECTIVE_LIMIT_MB:
        ax_ram.axvline(n_pairs[oom_idx], color=col, lw=1.3, linestyle=":")
        ax_ram.text(n_pairs[oom_idx] + 0.1, EFFECTIVE_LIMIT_MB/1024 - 0.3,
                    f"{name}\nOOM at n={n_pairs[oom_idx]}",
                    fontsize=7.5, color=col, fontweight="bold")

ax_ram.set_xlabel("Number of cached model instances", fontsize=10)
ax_ram.set_ylabel("Estimated RAM (GB)", fontsize=10)
ax_ram.set_title("RAM Usage vs Cached Model Count", fontsize=11)
ax_ram.legend(fontsize=8, loc="upper left")

# --- Bloom filter sizing chart ---
ax_bf = axes[1]
n_items_range = np.logspace(3, 7, 200)  # 1K to 10M items
for p_fpr, col, lbl in [(0.01, PALETTE_RED, "p=1%"),
                         (0.001, PALETTE_BLUE, "p=0.1%"),
                         (0.0001, "#3fb950", "p=0.01%")]:
    m_bits = -n_items_range * np.log(p_fpr) / (np.log(2)**2)
    ax_bf.plot(n_items_range, m_bits / 8 / 1024,
               color=col, lw=2.2, label=lbl)

ax_bf.axvline(BF_N, color="#f0e68c", lw=1.5, linestyle="--", label=f"n=1M (our config)")
ax_bf.scatter([BF_N], [bf_kb], color="#f0e68c", s=80, zorder=5)
ax_bf.text(BF_N * 1.5, bf_kb, f"{bf_kb:.1f} KB\n(n=1M, p=1%)",
           fontsize=8.5, color="#f0e68c", fontweight="bold")
ax_bf.set_xscale("log")
ax_bf.set_xlabel("Items n (log scale)", fontsize=10)
ax_bf.set_ylabel("Bloom filter size (KB)", fontsize=10)
ax_bf.set_title("Bloom Filter Memory: m = -n·ln(p) / (ln2)²", fontsize=11)
ax_bf.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "s7_memory_analysis.png"),
            bbox_inches="tight", dpi=150, facecolor=fig.get_facecolor())
plt.show()

# --- Report ---
deberta_oom = n_pairs[np.argmax(ram_deberta_only > EFFECTIVE_LIMIT_MB)]
toxic_oom   = n_pairs[np.argmax(ram_toxic_only > EFFECTIVE_LIMIT_MB)]
mixed_oom   = n_pairs[np.argmax(ram_mixed > EFFECTIVE_LIMIT_MB)]

print("\n=== Memory Consumption Report ===")
print(f"  Container OOM limit      : {CONTAINER_LIMIT_MB:,} MB ({CONTAINER_LIMIT_MB//1024} GB)")
print(f"  Safety margin reserved   : {SAFETY_MARGIN_MB} MB")
print(f"  Effective usable limit   : {EFFECTIVE_LIMIT_MB:,} MB")
print(f"  Python base footprint    : {PYTHON_BASE_MB} MB")
print(f"  DeBERTa-v3-base (FP32)   : {DEBERTA_MB:,} MB per instance")
print(f"  toxic-bert ONNX (INT8)   : {TOXIC_MB} MB per instance")
print()
print(f"  Max safe DeBERTa-only instances : {int(deberta_oom) - 1}")
print(f"  Max safe Toxic-only instances   : {int(toxic_oom) - 1}")
print(f"  Max safe mixed instances        : {int(mixed_oom) - 1}")
print()
print(f"  Bloom filter (n=1M, p=1%) : {bf_kb:.2f} KB = {bf_bits/8/1024/1024:.4f} MB")
print(f"  → Bloom filter is negligible vs model weights")

---
## Section 8 — SLO Compliance Dashboard

**SLO:** P95 request latency ≤ 150 ms, measured in 5-minute windows, over a 30-day period.
**Error budget:** 99.5% of 5-minute windows must be compliant → 0.5% breach budget = 2.16 hours/month.

**Simulation:** We inject:
- 3 degradation events (simulating upstream slowdowns)
- Gaussian noise on baseline latency
- Correct recovery after circuit breaker reset

In [ ]:
# ===========================================================================
# Section 8: SLO Compliance Dashboard
# WHY: The SLO is the contractual obligation. This section answers:
#      - Are we meeting it?
#      - How much error budget have we consumed?
#      - What is the Wilson 95% CI on the compliance rate?
#      - When do breaches cluster (shift patterns, deployment windows)?
# ===========================================================================

SLO_P95_MS   = 150.0   # SLO target in milliseconds
WINDOW_MIN   = 5       # measurement window in minutes
DAYS         = 30
REQ_PER_WIN  = 200     # approximate requests per 5-min window
WINDOWS_PER_DAY  = 24 * 60 // WINDOW_MIN   # = 288
TOTAL_WINDOWS    = WINDOWS_PER_DAY * DAYS  # = 8640

# Inject 3 degradation events (start_window, duration_windows, multiplier)
DEGRADATIONS = [
    (600,  48, 4.5),   # Day ~2: 4h degradation, 4.5× latency spike
    (2400, 24, 3.2),   # Day ~8: 2h degradation
    (6900, 72, 5.1),   # Day ~24: 6h severe degradation
]

baseline_mean_ms  = 28.0
baseline_sigma_ms = 4.0

def generate_window_p95(window_idx: int, n_req: int) -> float:
    """Generate P95 for one 5-minute measurement window."""
    mult = 1.0
    for start, dur, m in DEGRADATIONS:
        if start <= window_idx < start + dur:
            mult = m
            break
    sample = lognormal_from_mean_std(
        baseline_mean_ms * mult,
        baseline_sigma_ms * mult,
        n_req, rng
    )
    return float(np.percentile(sample, 95))

print("Generating 30-day SLO trace (8,640 windows × 200 req) ...")
p95_trace = np.array([generate_window_p95(i, REQ_PER_WIN)
                       for i in range(TOTAL_WINDOWS)])
print("Done.")

breach_mask    = p95_trace > SLO_P95_MS
compliant_mask = ~breach_mask
compliance_rate = compliant_mask.mean()
n_breaches = breach_mask.sum()
budget_consumed_pct = n_breaches / TOTAL_WINDOWS * 100
budget_pct = 0.5   # 99.5% compliance target → 0.5% error budget
budget_remaining_pct = max(0, budget_pct - budget_consumed_pct)

# Wilson CI on compliance rate
ci_lo, ci_hi = wilson_interval(TOTAL_WINDOWS, compliance_rate, Z_95)

# Rolling 1-day compliance rate (288 windows/day)
rolling_compliance = pd.Series(compliant_mask.astype(float)).rolling(
    WINDOWS_PER_DAY, min_periods=10).mean() * 100

window_times_days = np.arange(TOTAL_WINDOWS) / WINDOWS_PER_DAY

# --- Plots ---
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
fig.suptitle("SLO Compliance Dashboard — P95 ≤ 150 ms over 30-Day Window",
             fontsize=14, fontweight="bold", color="#c9d1d9")

# Plot 1: P95 latency trace
ax_p95 = axes[0]
ax_p95.plot(window_times_days, p95_trace, color="#8b949e", lw=0.6, alpha=0.5)
ax_p95.fill_between(window_times_days, p95_trace,
                    where=compliant_mask, color=PALETTE_BLUE, alpha=0.35,
                    label="Compliant windows")
ax_p95.fill_between(window_times_days, p95_trace,
                    where=breach_mask, color=PALETTE_RED, alpha=0.65,
                    label="SLO breach events")
ax_p95.axhline(SLO_P95_MS, color="#f0e68c", lw=2.0, linestyle="--",
               label=f"SLO target (P95 ≤ {SLO_P95_MS:.0f} ms)")
ax_p95.set_ylabel("P95 Latency (ms)", fontsize=10)
ax_p95.set_yscale("log")
ax_p95.legend(fontsize=9, loc="upper right")
ax_p95.set_title("Measured P95 Latency per 5-min Window", fontsize=11)

# Annotate degradation events
for start, dur, mult in DEGRADATIONS:
    day_start = start / WINDOWS_PER_DAY
    day_end   = (start + dur) / WINDOWS_PER_DAY
    ax_p95.axvspan(day_start, day_end, alpha=0.12, color=PALETTE_RED, zorder=0)
    ax_p95.text((day_start + day_end)/2, SLO_P95_MS * 3,
                f"×{mult}", ha="center", fontsize=8, color=PALETTE_RED)

# Plot 2: Rolling 24-h compliance rate
ax_comp = axes[1]
ax_comp.plot(window_times_days, rolling_compliance,
             color=PALETTE_BLUE, lw=1.8, label="Rolling 24-h compliance rate")
ax_comp.axhline(99.5, color="#f0e68c", lw=1.8, linestyle="--",
                label="SLO compliance target (99.5%)")
ax_comp.fill_between(window_times_days,
                     rolling_compliance, 99.5,
                     where=(rolling_compliance < 99.5),
                     color=PALETTE_RED, alpha=0.4, label="Budget burn")
ax_comp.set_ylabel("24-h Compliance (%)", fontsize=10)
ax_comp.set_ylim(80, 101)
ax_comp.legend(fontsize=9)

# Plot 3: Cumulative error budget burn
ax_bud = axes[2]
cumulative_burn = np.cumsum(breach_mask) / TOTAL_WINDOWS * 100
budget_limit = np.full(TOTAL_WINDOWS, budget_pct)
ax_bud.plot(window_times_days, cumulative_burn, color=PALETTE_RED, lw=2.0,
            label="Cumulative error budget consumed (%)")
ax_bud.plot(window_times_days, budget_limit, color="#f0e68c", lw=1.8,
            linestyle="--", label=f"Total error budget ({budget_pct}%)")
ax_bud.fill_between(window_times_days, cumulative_burn, budget_pct,
                    where=(cumulative_burn > budget_pct),
                    color=PALETTE_RED, alpha=0.5, label="Budget exceeded")
ax_bud.set_xlabel("Day", fontsize=10)
ax_bud.set_ylabel("Cumulative budget %", fontsize=10)
ax_bud.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "s8_slo_dashboard.png"),
            bbox_inches="tight", dpi=150, facecolor=fig.get_facecolor())
plt.show()

print("\n=== SLO Compliance Report ===")
print(f"  Measurement period  : {DAYS} days")
print(f"  Total 5-min windows : {TOTAL_WINDOWS:,}")
print(f"  SLO target          : P95 ≤ {SLO_P95_MS:.0f} ms (99.5% window compliance)")
print(f"  Compliant windows   : {compliant_mask.sum():,} ({compliance_rate*100:.3f}%)")
print(f"  Breach windows      : {n_breaches:,} ({n_breaches/TOTAL_WINDOWS*100:.3f}%)")
print(f"  Wilson 95% CI       : [{ci_lo*100:.3f}%, {ci_hi*100:.3f}%]")
print(f"  Error budget total  : {budget_pct}% ({budget_pct/100*DAYS*24:.2f} hours/month)")
print(f"  Budget consumed     : {budget_consumed_pct:.3f}%")
print(f"  Budget remaining    : {budget_remaining_pct:.3f}%")
print(f"  Status              : {'⚠ BUDGET EXCEEDED' if budget_consumed_pct > budget_pct else '✓ Within budget'}")

---
## Section 9 — Statistical Summary Table

Consolidated reference table for all key metrics, suitable for the ADR appendix or on-call runbook.

In [ ]:
# ===========================================================================
# Section 9: Statistical Summary Table
# WHY: Every finding from this notebook needs a single-page summary that
#      an on-call engineer can reference at 3 AM. This table is that page.
# ===========================================================================

summary_rows = [
    # --- Image Footprint ---
    {"Section": "S1", "Metric": "CPU image size (baked)",
     "Value": f"{BAKED_MB[0]:,} MB", "Interpretation": "Baseline"},
    {"Section": "S1", "Metric": "CPU image size (decoupled)",
     "Value": f"{DECOUPLE_MB[0]:,} MB",
     "Interpretation": f"−{reductions_pct[0]:.1f}% size reduction"},
    {"Section": "S1", "Metric": "GPU image size (baked)",
     "Value": f"{BAKED_MB[1]:,} MB", "Interpretation": "Baseline"},
    {"Section": "S1", "Metric": "GPU image size (decoupled)",
     "Value": f"{DECOUPLE_MB[1]:,} MB",
     "Interpretation": f"−{reductions_pct[1]:.1f}% size reduction"},
    {"Section": "S1", "Metric": "6-month CPU pull-time saved",
     "Value": f"{s_to_h(cumulative_baked_cpu[-1] - cumulative_decoup_cpu[-1]):.1f} h",
     "Interpretation": "At 100 MB/s, 60 deploys/month"},
    # --- Latency ---
    {"Section": "S2", "Metric": "Cold start P50",
     "Value": f"{np.percentile(cold_start, 50):.0f} ms",
     "Interpretation": "Dynamic download path"},
    {"Section": "S2", "Metric": "Cold start P95",
     "Value": f"{np.percentile(cold_start, 95):.0f} ms",
     "Interpretation": "Tail latency unacceptable"},
    {"Section": "S2", "Metric": "Warm cache P95",
     "Value": f"{np.percentile(warm_cache, 95):.1f} ms",
     "Interpretation": "Within SLO"},
    {"Section": "S2", "Metric": "Welch's t p-value (cold vs warm)",
     "Value": f"{p_val:.2e}",
     "Interpretation": "Null REJECTED at α=0.001"},
    {"Section": "S2", "Metric": "Cohen's d (cold vs warm)",
     "Value": f"{d:,.1f}",
     "Interpretation": "'Huge' effect (>>2)"},
    # --- Token Analysis ---
    {"Section": "S3", "Metric": "Single-pass requests (≤510 tokens)",
     "Value": f"{pct_single:.2f}%",
     "Interpretation": "No dual-pass overhead"},
    {"Section": "S3", "Metric": "Dual-pass requests (>510 tokens)",
     "Value": f"{pct_dual:.2f}%",
     "Interpretation": "2× latency minimum"},
    {"Section": "S3", "Metric": "P95 latency uplift from dual-pass",
     "Value": f"+{uplift_pct:.2f}%",
     "Interpretation": "vs pure single-pass baseline"},
    # --- Algorithms ---
    {"Section": "S4", "Metric": "EWMA α=0.3 convergence speed",
     "Value": f"{dict(convergence_report)[0.3]} samples",
     "Interpretation": "90% of step-change captured"},
    {"Section": "S4", "Metric": "Bloom FPR at n=5000 (m=50K, k=3)",
     "Value": f"{bloom_fpr_theoretical(5000, M_FILTER, K_HASH)*100:.3f}%",
     "Interpretation": "Below 1% threshold"},
    {"Section": "S4", "Metric": "Min-heap speedup at n=10K",
     "Value": f"{sortedlist_times[-1]/heap_times[-1]:.1f}×",
     "Interpretation": "vs sorted list insertion"},
    {"Section": "S4", "Metric": "CMS error at w=2000, d=3",
     "Value": f"{cms_results[3][WIDTHS.index(2000)]:.3f}%",
     "Interpretation": "Mean relative error on top-100"},
    {"Section": "S4", "Metric": "Min n for 80% power (10% toxicity shift)",
     "Value": str(min_n_80),
     "Interpretation": "Per group, α=0.05"},
    # --- Concurrency ---
    {"Section": "S5", "Metric": "Lock contention rate",
     "Value": f"{len(downloads)/N_THREADS*100:.1f}%" if downloads else "N/A",
     "Interpretation": "Threads blocked on model lock"},
    # --- Circuit Breaker ---
    {"Section": "S6", "Metric": "CB trips in 120 requests",
     "Value": str(n_trips),
     "Interpretation": f"At {FAILURE_RATE*100:.0f}% failure rate"},
    {"Section": "S6", "Metric": "Mean time to recovery",
     "Value": f"{mttr:.1f} requests",
     "Interpretation": "OPEN → CLOSED cycle"},
    {"Section": "S6", "Metric": "Cascade failures avoided",
     "Value": str(avoided_failures),
     "Interpretation": "Failures shunted while OPEN"},
    # --- Memory ---
    {"Section": "S7", "Metric": "Max safe DeBERTa instances",
     "Value": str(int(deberta_oom) - 1),
     "Interpretation": f"In {CONTAINER_LIMIT_MB//1024} GB container"},
    {"Section": "S7", "Metric": "Bloom filter size (n=1M, p=1%)",
     "Value": f"{bf_kb:.1f} KB",
     "Interpretation": "Negligible overhead"},
    # --- SLO ---
    {"Section": "S8", "Metric": "30-day SLO compliance rate",
     "Value": f"{compliance_rate*100:.3f}%",
     "Interpretation": f"Wilson 95% CI: [{ci_lo*100:.2f}%, {ci_hi*100:.2f}%]"},
    {"Section": "S8", "Metric": "Error budget consumed",
     "Value": f"{budget_consumed_pct:.3f}%",
     "Interpretation": f"of {budget_pct}% total budget"},
]

df_summary = pd.DataFrame(summary_rows)

# Pretty-print Markdown table
print("\n" + "=" * 85)
print("  RESEARCH SUMMARY TABLE — NLI Decoupling ADR-0041")
print("=" * 85)
print(df_summary.to_markdown(index=False, tablefmt="pipe"))
print()

# Also save as CSV for ADR appendix import
csv_path = os.path.join(OUTPUT_DIR, "summary_metrics.csv")
df_summary.to_csv(csv_path, index=False)
print(f"Summary table saved to: {csv_path}")
print(f"All charts saved to:    {OUTPUT_DIR}/")
print()
print("=" * 85)
print("  RESEARCH COMPLETE — All 8 Research Questions Answered")
print("=" * 85)